In [27]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


# Locate the project root
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "public"
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMAGES = DATA_DIR / "train"
TEST_IMAGES = DATA_DIR / "test"

assert TRAIN_CSV.exists(), f"Missing CSV: {TRAIN_CSV}"
assert TRAIN_IMAGES.is_dir(), f"Missing training images: {TRAIN_IMAGES}"

df = pd.read_csv(TRAIN_CSV)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Unique image IDs:", df["image_id"].nunique())
print("Defect classes:", sorted(df["defect_type"].dropna().unique()))
print("Training images:", len(list(TRAIN_IMAGES.glob("*.jpg"))))
print("Test images:", len(list(TEST_IMAGES.glob("*.jpg"))))

display(df.head())


Dataset shape: (45572, 5)
Columns: ['image_id', 'defect_type', 'mask_rle', 'width', 'height']
Unique image IDs: 11393
Defect classes: ['defect_a', 'defect_b', 'defect_c', 'defect_d']
Training images: 11393
Test images: 3000


,image_id,defect_type,mask_rle,width,height
0,4S22I7SA34,defect_a,7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...,800,128
1,4S22I7SA34,defect_b,NaN,800,128
2,4S22I7SA34,defect_c,NaN,800,128
3,4S22I7SA34,defect_d,NaN,800,128
4,DZ899111OE,defect_a,NaN,800,128


In [28]:
import math

# 1. Bottleneck dimensions for 128 x 800
height, width = 128, 800

for _ in range(4):
    height //= 2
    width //= 2

print("Bottleneck shape:", (height, width))

# 2. Odd-width dimension mismatch
odd_width = 799
encoder_width = odd_width

for _ in range(4):
    encoder_width //= 2

decoder_width = encoder_width * 16
mismatch = odd_width - decoder_width

print("Odd-width mismatch:", mismatch)

# 3. Spatial downsampling factor
input_area = 128 * 800
bottleneck_area = height * width

downsample_factor = input_area // bottleneck_area

print("Downsampling factor:", downsample_factor)

Bottleneck shape: (8, 50)
Odd-width mismatch: 15
Downsampling factor: 256


In [29]:
import segmentation_models_pytorch as smp

def audit_model(encoder_name):
    model = smp.Unet(
        encoder_name=encoder_name,
        in_channels=1,
        classes=4
    )

    total_params = sum(p.numel() for p in model.parameters())

    encoder_params = sum(
        p.numel() for p in model.encoder.parameters()
    )

    trainable_params = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

    # Freeze encoder to simulate feature-extractor transfer learning
    for p in model.encoder.parameters():
        p.requires_grad = False

    remaining_trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

    return total_params, encoder_params, remaining_trainable


# Model A: ResNet-34
resnet_total, resnet_encoder, resnet_trainable = audit_model("resnet34")

# Model B: EfficientNet-B0
efficientnet_total, _, _ = audit_model("efficientnet-b0")

# Parameter comparison
difference = abs(resnet_total - efficientnet_total)

# Percentage remaining trainable for ResNet-34 U-Net
trainable_pct = (resnet_trainable / resnet_total) * 100

print("ResNet-34 total parameters:", resnet_total)
print("ResNet-34 encoder parameters:", resnet_encoder)
print("ResNet-34 trainable after freezing encoder:", resnet_trainable)

print("EfficientNet-B0 total parameters:", efficientnet_total)
print("Parameter difference:", difference)
print("Trainable percentage:", round(trainable_pct, 1))

print(
    "Final answer:",
    f"{'EfficientNet-B0' if efficientnet_total < resnet_total else 'ResNet-34'},",
    f"{difference}; {trainable_pct:.1f}%"
)

ResNet-34 total parameters: 24430532
ResNet-34 encoder parameters: 21278400
ResNet-34 trainable after freezing encoder: 3152132
EfficientNet-B0 total parameters: 6251328
Parameter difference: 18179204
Trainable percentage: 12.9
Final answer: EfficientNet-B0, 18179204; 12.9%


In [30]:
import torch
import torch.nn.functional as F

# Simulate pretrained RGB convolution weights
torch.manual_seed(42)

W_rgb = torch.randn(8, 3, 3, 3)
bias = torch.randn(8)

# Grayscale input
I = torch.randn(1, 1, 128, 800)

# Strategy A: Duplicate grayscale input into 3 channels
X_rep = I.repeat(1, 3, 1, 1)
output_A = F.conv2d(X_rep, W_rgb, bias, padding=1)

# Strategy B: Collapse RGB weights using SUM
W_gray = W_rgb.sum(dim=1, keepdim=True)
output_B = F.conv2d(I, W_gray, bias, padding=1)

# Verify identical activations
print("Outputs identical:", torch.allclose(output_A, output_B, atol=1e-5))

# Verify input memory reduction
strategy_A_size = X_rep.numel()
strategy_B_size = I.numel()

print("Strategy A input elements:", strategy_A_size)
print("Strategy B input elements:", strategy_B_size)
print("Memory reduction factor:", strategy_A_size // strategy_B_size)

Outputs identical: True
Strategy A input elements: 307200
Strategy B input elements: 102400
Memory reduction factor: 3


In [31]:
# Question 4: Effective Receptive Field Calculation

kernel_size = 3
dilation_rates = [6, 12]

for rate in dilation_rates:
    effective_size = kernel_size + (kernel_size - 1) * (rate - 1)
    print(
        f"Dilation rate = {rate}: "
        f"Effective footprint = {effective_size} x {effective_size}"
    )



Dilation rate = 6: Effective footprint = 13 x 13
Dilation rate = 12: Effective footprint = 25 x 25


In [32]:
# Question 5: Soft Dice Loss Calculation

epsilon = 1.0

total_pixels = 10000
defect_pixels = 100
background_pixels = 9900

# Model 1: Uniform prediction of 0.01
p1_sum = total_pixels * 0.01
intersection1 = defect_pixels * 0.01
y_sum = defect_pixels

loss_model1 = 1 - (
    (2 * intersection1 + epsilon) /
    (p1_sum + y_sum + epsilon)
)

# Model 2: Prediction of 0.90 on defects and 0.00 on background
p2_sum = defect_pixels * 0.90
intersection2 = defect_pixels * 0.90

loss_model2 = 1 - (
    (2 * intersection2 + epsilon) /
    (p2_sum + y_sum + epsilon)
)

print(f"Model 1 Soft Dice Loss: {loss_model1:.4f}")
print(f"Model 2 Soft Dice Loss: {loss_model2:.4f}")




Model 1 Soft Dice Loss: 0.9851
Model 2 Soft Dice Loss: 0.0524


In [33]:
# Five output channels: Background + 4 defect classes
logits = torch.zeros(1, 5, 1, 4)

# Left half: defect_a, right half: defect_c
logits[0, 1, 0, :2] = 5.0
logits[0, 3, 0, 2:] = 5.0

# Apply Softmax and select the most likely class per pixel
probabilities = torch.softmax(logits, dim=1)
predictions = torch.argmax(probabilities, dim=1)

print("Predicted classes:", predictions)

Predicted classes: tensor([[[1, 1, 3, 3]]])


In [34]:
# Question 7: Soft Dice Loss on a clean image

epsilon = 1.0
actual_defects = 0

# Case 1: No predicted defects
predicted_pixels_clean = 0

dice_clean = (
    2 * 0 + epsilon
) / (
    predicted_pixels_clean + actual_defects + epsilon
)

loss_clean = 1 - dice_clean

# Case 2: 50 false-positive predictions
predicted_pixels_noise = 50

dice_noise = (
    2 * 0 + epsilon
) / (
    predicted_pixels_noise + actual_defects + epsilon
)

loss_noise = 1 - dice_noise

print(f"Dice loss with clean predictions: {loss_clean:.4f}")
print(f"Dice loss with 50 false positives: {loss_noise:.4f}")

Dice loss with clean predictions: 0.0000
Dice loss with 50 false positives: 0.9804


In [35]:
%pip install -q segmentation-models-pytorch opencv-python-headless

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Project paths
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "public"
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMAGES = DATA_DIR / "train"

df = pd.read_csv(TRAIN_CSV)

CLASS_NAMES = ["defect_a", "defect_b", "defect_c", "defect_d"]
NUM_CLASSES = len(CLASS_NAMES)

HEIGHT = 128
WIDTH = 800

# Locate image files by image ID, regardless of extension
image_files = {
    p.stem: p for p in TRAIN_IMAGES.iterdir()
    if p.is_file()
}

# Keep only image IDs that have corresponding image files
image_ids = [
    image_id for image_id in df["image_id"].unique()
    if str(image_id) in image_files
]

# Use the same split for both models
train_ids, val_ids = train_test_split(
    image_ids,
    test_size=0.2,
    random_state=SEED
)

print("Training images:", len(train_ids))
print("Validation images:", len(val_ids))
print("Classes:", CLASS_NAMES)


def decode_rle(rle, height, width):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask.reshape((height, width), order="F")

    values = np.asarray(str(rle).split(), dtype=int)
    starts = values[0::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):
        mask[start:start + length] = 1

    return mask.reshape((height, width), order="F")


class SteelDefectDataset(Dataset):

    def __init__(self, image_ids, dataframe):
        self.image_ids = list(image_ids)
        self.df = dataframe
        self.grouped = {
            image_id: group.set_index("defect_type")
            for image_id, group in dataframe.groupby("image_id")
        }

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]

        image_path = image_files[str(image_id)]

        image = Image.open(image_path).convert("L")
        image = image.resize((WIDTH, HEIGHT))

        image = np.asarray(image, dtype=np.float32) / 255.0

        # Same normalization for both models
        image = (image - 0.449) / 0.226

        image = torch.from_numpy(image).unsqueeze(0)

        masks = np.zeros(
            (NUM_CLASSES, HEIGHT, WIDTH),
            dtype=np.float32
        )

        image_rows = self.grouped[image_id]

        for class_index, class_name in enumerate(CLASS_NAMES):
            if class_name in image_rows.index:
                rle = image_rows.loc[class_name, "mask_rle"]

                masks[class_index] = decode_rle(
                    rle, HEIGHT, WIDTH
                )

        masks = torch.from_numpy(masks)

        return image, masks


train_dataset = SteelDefectDataset(train_ids, df)
val_dataset = SteelDefectDataset(val_ids, df)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("DataLoaders created successfully.")

Training images: 9114
Validation images: 2279
Classes: ['defect_a', 'defect_b', 'defect_c', 'defect_d']
DataLoaders created successfully.


In [37]:
class CompoundLoss(nn.Module):

    def __init__(self, alpha=0.5, beta=0.5, smooth=1.0):
        super().__init__()

        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):

        bce_loss = self.bce(logits, targets)

        probabilities = torch.sigmoid(logits)

        intersection = (probabilities * targets).sum(
            dim=(2, 3)
        )

        denominator = (
            probabilities.sum(dim=(2, 3))
            + targets.sum(dim=(2, 3))
        )

        dice = (
            2 * intersection + self.smooth
        ) / (
            denominator + self.smooth
        )

        dice_loss = 1 - dice.mean()

        return (
            self.alpha * bce_loss
            + self.beta * dice_loss
        )

In [38]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


def create_model(pretrained):

    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet" if pretrained else None,
        in_channels=1,
        classes=4
    )

    return model.to(device)


scratch_model = create_model(pretrained=False)
pretrained_model = create_model(pretrained=True)

print("Both models created successfully.")

Using device: cpu
Both models created successfully.


In [39]:
criterion = CompoundLoss(
    alpha=0.5,
    beta=0.5,
    smooth=1.0
)


def calculate_foreground_dice(model, loader):

    model.eval()

    intersection = torch.zeros(NUM_CLASSES, device=device)
    pred_sum = torch.zeros(NUM_CLASSES, device=device)
    target_sum = torch.zeros(NUM_CLASSES, device=device)

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            predictions = (torch.sigmoid(logits) >= 0.5).float()

            intersection += (predictions * masks).sum(
                dim=(0, 2, 3)
            )

            pred_sum += predictions.sum(dim=(0, 2, 3))
            target_sum += masks.sum(dim=(0, 2, 3))

    dice = (
        2 * intersection + 1.0
    ) / (
        pred_sum + target_sum + 1.0
    )

    return dice.mean().item()


def run_epoch(model, loader, optimizer=None):

    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0

    for images, masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):

            logits = model(images)
            loss = criterion(logits, masks)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


def train_model(model, model_name):

    optimizer = optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=1e-2
    )

    history = []

    for epoch in range(1, 6):

        train_loss = run_epoch(
            model, train_loader, optimizer
        )

        val_loss = run_epoch(
            model, val_loader
        )

        foreground_dice = calculate_foreground_dice(
            model, val_loader
        )

        result = {
            "model": model_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "foreground_dice": foreground_dice
        }

        history.append(result)

        print(
            f"{model_name} | Epoch {epoch}/5 | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Foreground Dice: {foreground_dice:.4f}"
        )

    return history

In [ ]:
scratch_history = train_model(
    scratch_model,
    "Scratch U-Net"
)

pretrained_history = train_model(
    pretrained_model,
    "Pretrained ResNet-34 U-Net"
)

results = pd.DataFrame(
    scratch_history + pretrained_history
)

print("\nComplete training results:")
display(results)

In [ ]:
scratch_results = results[
    results["model"] == "Scratch U-Net"
].reset_index(drop=True)

pretrained_results = results[
    results["model"] == "Pretrained ResNet-34 U-Net"
].reset_index(drop=True)

comparison = pd.DataFrame({
    "Epoch": range(1, 6),
    "Scratch Train Loss": scratch_results["train_loss"],
    "Scratch Val Loss": scratch_results["val_loss"],
    "Scratch Dice": scratch_results["foreground_dice"],
    "Pretrained Train Loss": pretrained_results["train_loss"],
    "Pretrained Val Loss": pretrained_results["val_loss"],
    "Pretrained Dice": pretrained_results["foreground_dice"]
})

display(comparison)

# Epoch 5 comparison
scratch_final = scratch_results.iloc[-1]
pretrained_final = pretrained_results.iloc[-1]

scratch_gap = (
    scratch_final["val_loss"] - scratch_final["train_loss"]
)

pretrained_gap = (
    pretrained_final["val_loss"] - pretrained_final["train_loss"]
)

print("\nEpoch 5 comparison")

print("Scratch validation loss:", round(scratch_final["val_loss"], 4))
print("Pretrained validation loss:", round(pretrained_final["val_loss"], 4))

print("Scratch foreground Dice:", round(scratch_final["foreground_dice"], 4))
print("Pretrained foreground Dice:", round(pretrained_final["foreground_dice"], 4))

print("\nEpoch 1 foreground Dice")

print("Scratch:", round(scratch_results.iloc[0]["foreground_dice"], 4))
print("Pretrained:", round(pretrained_results.iloc[0]["foreground_dice"], 4))

print("\nGeneralization gaps")

print("Scratch:", round(scratch_gap, 4))
print("Pretrained:", round(pretrained_gap, 4))

print(
    "\nSmaller generalization gap:",
    "Pretrained" if abs(pretrained_gap) < abs(scratch_gap)
    else "Scratch"
)

NameError: name 'results' is not defined

In [ ]:
# Question 9: ResNet-34 vs EfficientNet-B0

resnet_params = 24430532
efficientnet_params = 6251328

# Calculate parameter ratio
parameter_ratio = resnet_params / efficientnet_params

# Calculate percentage reduction
reduction = (
    (resnet_params - efficientnet_params)
    / resnet_params
) * 100

print("ResNet-34 parameters:", f"{resnet_params:,}")
print("EfficientNet-B0 parameters:", f"{efficientnet_params:,}")

print("Parameter ratio:", round(parameter_ratio))
print("Parameter reduction:", f"{reduction:.1f}%")

print("EfficientNet convolution technique: Depthwise separable convolution")

ResNet-34 parameters: 24,430,532
EfficientNet-B0 parameters: 6,251,328
Parameter ratio: 4
Parameter reduction: 74.4%
EfficientNet convolution technique: Depthwise separable convolution
